In [11]:
import os
from huggingface_hub import hf_hub_download, list_repo_files
import os, glob
import pyarrow as pa
import pyarrow.parquet as pq

In [6]:
DATASET_DIR = "/kaggle/working/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)



In [7]:
files = [
    f for f in files
    if f.startswith("big-multilabel/") and f.endswith(".parquet")
]

In [ ]:
for i, f in enumerate(files, 1):
    print(f"Downloading {i}/{len(files)}: {f}")

    hf_hub_download(
        repo_id="mwritescode/slither-audited-smart-contracts",
        filename=f,
        repo_type="dataset",
        revision="refs/convert/parquet",
        token="************************************",
        local_dir=DATASET_DIR
    )

print("\nDownload complete!")
print("Files saved in:", DATASET_DIR)


Download complete!
Files saved in: /kaggle/working/dataset


In [9]:

DATASET_DIR = "/kaggle/working/dataset"
CONFIG = "big-multilabel"
OUT_DIR = "/kaggle/working/merged"
os.makedirs(OUT_DIR, exist_ok=True)

SPLITS = {"train": "train_dataset.parquet",
          "test": "test_dataset.parquet",
          "validation": "val_dataset.parquet"}


In [12]:
shards = {sp: sorted(glob.glob(
              os.path.join(DATASET_DIR, CONFIG, sp, "*.parquet")))
          for sp in SPLITS}
for sp, ps in shards.items():
    print(sp.ljust(11), len(ps), "shard(s)")
    assert ps, f"no shards found for {sp} - check DATASET_DIR / CONFIG"

train       7 shard(s)
test        2 shard(s)
validation  1 shard(s)


In [18]:
counts = {}
for sp, out_name in SPLITS.items():
    out = os.path.join(OUT_DIR, out_name)
    writer, n = None, 0
    for p in shards[sp]:
        pf = pq.ParquetFile(p)
        if writer is None:
            writer = pq.ParquetWriter(out, pf.schema_arrow,
                                      compression="snappy")
        for g in range(pf.num_row_groups):
            tbl = pf.read_row_group(g)
            writer.write_table(tbl)
            n += tbl.num_rows
    writer.close()
    counts[sp] = n



print("\ncolumns:", pq.read_schema(out).names)
print("rows:", counts,)



columns: ['address', 'source_code', 'bytecode', 'slither']
rows: {'train': 79641, 'test': 15972, 'validation': 10861}
